In [0]:
# =============================================================================
# NOTEBOOK  : silver_pos_transactions_batch
# PURPOSE   : bronze.pos_transactions (batch rows) → silver.pos_transactions
# LAYER     : Silver (clean, type-cast, merge, delete cancelled transactions)
# FREQUENCY : Daily at 2 AM (JOB B)
# TRIGGER   : availableNow
#
# MERGE & DEDUP LOGIC:
#   Source  : bronze.pos_transactions WHERE _source = 'pos_batch_csv'
#   No within-batch dedup needed:
#     Assumption: batch CSV is clean and deduplicated by source system
#     No two records in batch have same transaction_id
#
#   MERGE cases:
#     Case 1: transaction_id IN silver, no data change  → IGNORE
#     Case 2: transaction_id NOT in silver              → INSERT
#     Case 3: transaction_id IN silver, data changed    → UPDATE
#             Changeable: total_amount, quantity, payment_method, channel
#             transaction_ts NOT updated — stream event time is authoritative
#             source_system NOT updated — preserves original stream source
#
# ── DELETE CANCELLED TRANSACTIONS (Case 4) ────────────────────────────
# Scope: only transaction_dates that are actually present in batch file
# These were stream-inserted but batch confirms they were cancelled.
# Uses anti-join — efficient for large sets vs NOT IN
# This handles:
#   - Single day batch files
#   - Multi-day batch files
#   - Batch files with gaps (Day 1,2,4,5 but not Day 3)
# Day 3 records are untouched because Day 3 is not in batch_dates
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, to_timestamp,
    min as spark_min, max as spark_max
)
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_pos_transactions"]
TARGET_TABLE = cfg["tables"]["silver_pos_transactions"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_pos_transactions_batch"]
PIPELINE     = "silver_pos_transactions_batch"

In [0]:

 
# ── foreachBatch function + Stream ────────────────────────────────────
 
def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze rows.
 
    FILTER   : Only batch rows (_source = pos_batch_csv)
    NO DEDUP : Batch CSV guaranteed clean by source system
    MERGE    : 
        Case 1 → record already present in silver table → IGNORE, 
        Case 2 → record not present in silver table → INSERT, 
        Case 3 → record present in silver, but some updates → UPDATE
    DELETE   : Case 4 → delete cancelled transactions scoped to batch date range
    """
    # ── executor-side imports ─────────────────────────────────
    import sys
    sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA

    # ── FILTER — batch rows only ──────────────────────────────────────────────
    micro_batch_df = micro_batch_df.filter(
        col("_source") == "pos_batch_csv"
    )
 
    if micro_batch_df.isEmpty():
        return
 
    run_id = start_audit(spark, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
    try:
        rows_read = micro_batch_df.count()
 
        # ── TRANSFORM ─────────────────────────────────────────────────────────
        df = (
            micro_batch_df

            # 1. Cast timestamp ISO string → TimestampType
            #    Source format: "2023-08-19T22:26:11Z"
            #    Renamed to transaction_ts — avoids Spark reserved word 'timestamp'
            .withColumn("transaction_ts",   to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))

            # 2. Money columns → Decimal for precision
            #    Double has floating point imprecision — critical for financial data
            .withColumn("unit_price",       col("unit_price").cast(DecimalType(10, 2)))
            .withColumn("total_amount",     col("total_amount").cast(DecimalType(10, 2)))

            # 3. transaction_date derived from event time (transaction_ts)
            #    NOT from ingested_at — silver partitions by when event happened
            .withColumn("transaction_date", to_date(col("transaction_ts")).cast("string"))

            # 4. Silver audit columns
            .withColumn("processed_at",     current_timestamp())
            .withColumn("pipeline_run_id",  lit(run_id))
            .withColumn("source_system",    lit("pos_batch_csv"))

            # 5. Enforce silver schema — drops bronze-only cols in one line
            #    (_source, source_file, ingested_date, etc.)
            .select([f.name for f in SILVER_POS_TRANSACTIONS_SCHEMA.fields])
        )
 
        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Case 3: matched + data changed → UPDATE changeable fields only
        # Case 2: not matched            → INSERT
        # Case 1: matched + no change    → no rule fires → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                "t.transaction_id = s.transaction_id"
            )
            .whenMatchedUpdate(
                condition="""
                    t.total_amount    != s.total_amount    OR
                    t.quantity        != s.quantity         OR
                    t.payment_method  != s.payment_method   OR
                    t.channel         != s.channel
                """,
                set={
                    "total_amount":    "s.total_amount",
                    "quantity":        "s.quantity",
                    "payment_method":  "s.payment_method",
                    "channel":         "s.channel",
                    "processed_at":    "current_timestamp()",
                    "pipeline_run_id": f"'{run_id}'"
                    # transaction_ts — NOT updated, stream event time is authoritative
                    # source_system  — NOT updated, preserves original stream source
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
 
        merge_metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(merge_metrics.get("numTargetRowsInserted", 0))
        rows_updated  = int(merge_metrics.get("numTargetRowsUpdated", 0))
 
        # ── DELETE CANCELLED TRANSACTIONS (Case 4) # ── DELETE CANCELLED TRANSACTIONS (Case 4) ────────────────────────────
        # Scope: only transaction_dates that are actually present in batch file
        # This handles:
        #   - Single day batch files
        #   - Multi-day batch files
        #   - Batch files with gaps (Day 1,2,4,5 but not Day 3)
        # Day 3 records are untouched because Day 3 is not in batch_dates

        # Exact dates present in this batch file
        batch_dates    = df.select("transaction_date").distinct()
        batch_txn_ids  = df.select("transaction_id").distinct()

        # Silver records whose date is covered by this batch
        silver_in_scope = (
            spark.read.table(TARGET_TABLE)
            .join(batch_dates, "transaction_date", "inner")
            .select("transaction_id", "transaction_date")
        )

        # Records in silver for those dates but NOT in batch = cancelled
        cancelled_txns = silver_in_scope.join(
            batch_txn_ids, "transaction_id", "left_anti"
        )

        rows_to_delete = cancelled_txns.count()
        rows_deleted   = 0

        if rows_to_delete > 0:
            cancelled_ids = [r.transaction_id for r in cancelled_txns.collect()]

            (
                DeltaTable.forName(spark, TARGET_TABLE)
                .delete(
                    col("transaction_id").isin(cancelled_ids) &
                    col("transaction_date").isin(
                        [r.transaction_date for r in batch_dates.collect()]
                    )
                )
            )

            delete_metrics = (
                spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
                .select("operationMetrics")
                .collect()[0][0]
            )
            rows_deleted = int(delete_metrics.get("numDeletedRows", 0))
 
        rows_written = rows_inserted + rows_updated
 
        end_audit(
            spark, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_deleted,   # rows_rejected tracks deletes in audit
            extra={
                "rows_inserted": str(rows_inserted),
                "rows_updated":  str(rows_updated),
                "rows_deleted":  str(rows_deleted)   # for batch notebook
                "microbatch_id": str(microbatch_id)
            }
        )
 
    except Exception as e:
        end_audit(spark, run_id, "FAILED", error=str(e))
        raise
 
# ── READ + WRITE ──────────────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)
 
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)
 
query.awaitTermination()

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
# Run this cell manually after the stream completes to verify the run

# ── 1. Latest audit entry for this pipeline ───────────────────────────────────
print("=" * 60)
print("AUDIT LOG — Last 3 runs")
print("=" * 60)
(
    spark.read.table("azure3_team7_project.platform.pipeline_audit")
    .filter(col("pipeline_name") == PIPELINE)
    .orderBy(col("start_time").desc())
    .limit(3)
    .select(
        "run_id", "status", "rows_read",
        "rows_written", "rows_rejected",
        "start_time", "end_time", "error_message"
    )
    .show(truncate=False)
)

# ── 2. Row counts in silver by source_system ──────────────────────────────────
print("=" * 60)
print("SILVER TABLE — Row counts by source_system")
print("=" * 60)
(
    spark.read.table(TARGET_TABLE)
    .groupBy("source_system")
    .count()
    .show(truncate=False)
)

# ── 3. Row counts by transaction_date ─────────────────────────────────────────
print("=" * 60)
print("SILVER TABLE — Row counts by transaction_date (last 5 dates)")
print("=" * 60)
(
    spark.read.table(TARGET_TABLE)
    .groupBy("transaction_date")
    .count()
    .orderBy(col("transaction_date").desc())
    .limit(5)
    .show(truncate=False)
)

# ── 4. Sample rows from latest pipeline_run_id ────────────────────────────────
print("=" * 60)
print("SILVER TABLE — Sample rows from latest run")
print("=" * 60)
latest_run_id = (
    spark.read.table("azure3_team7_project.platform.pipeline_audit")
    .filter(col("pipeline_name") == PIPELINE)
    .filter(col("status") == "SUCCESS")
    .orderBy(col("start_time").desc())
    .limit(1)
    .select("run_id")
    .collect()[0][0]
)
print(f"Latest successful run_id: {latest_run_id}")
(
    spark.read.table(TARGET_TABLE)
    .filter(col("pipeline_run_id") == latest_run_id)
    .limit(5)
    .select(
        "transaction_id", "store_id", "transaction_ts",
        "total_amount", "source_system", "transaction_date"
    )
    .show(truncate=False)
)

# ── 5. Delta history — last 3 operations on silver table ──────────────────────
print("=" * 60)
print("DELTA HISTORY — Last 3 operations on silver table")
print("=" * 60)
(
    spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 3")
    .select(
        "version", "timestamp", "operation",
        "operationMetrics", "userName"
    )
    .show(truncate=False)
)